# 02 — Earnings-call facts: where the disclosure comes from

Every realized `EARNINGS_RELEASE` event carries a `disclosure` block, and inside it an
item with `kind="facts"` / `source="earnings_call"` whose `content` is **ten sentences**
distilled from the company's earnings call. Those ten sentences are the substantive
input most agents build on, so it is worth knowing exactly how they are produced.

This notebook walks the whole path:

1. What you receive — pull the facts out of an archive record
2. The artifact behind the pointer
3. What the model actually saw — the rendered transcript
4. The exact prompt, printed in full
5. The request surface — and what is deliberately *not* in it
6. How extraction failures surface (`parse_note`)
7. Optionally, run the real call yourself

`examples.summary` is an exact semantic port of the production summarizer, so the
request built here is the request the pipeline sends.

> **You cannot regenerate a competition disclosure with this.** The production input is
> a licensed full-length transcript that is not distributed with this repo, and extended
> thinking makes the call nondeterministic even on identical input. Everything below runs
> on a small **synthetic** transcript for a fictional company.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from examples import load_config
from examples.archive import read_jsonl_gz
from examples.summary import (
    MODEL_CONFIGS,
    N_FACTS,
    SUMMARY_MODEL,
    build_request,
    facts_from_disclosure,
    format_transcript,
    recover_facts,
    summarize_transcript,
)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
SAMPLE_DIR = REPO / "data" / "sample"

config = load_config()
print("Summarizer demo:", "LIVE" if config.has_anthropic else "OFF (no ANTHROPIC_API_KEY)")
print("Facts per summary:", N_FACTS)
print("Production model:", SUMMARY_MODEL)

## 1. What you receive

Start from the delivered side. Each archive line is an event payload; the facts live at
`disclosure.items[]` under the entry whose `kind` is `facts` and whose `source` is
`earnings_call`. `facts_from_disclosure` does that lookup and returns `[]` when an event
has no such item — a scheduled-but-not-yet-occurred event legitimately has none, and so
does a call whose extraction failed.

(The bundled sample is illustrative and carries fewer than ten facts; real archive lines
carry ten.)

In [ ]:
records = list(read_jsonl_gz(SAMPLE_DIR / "archive_EARNINGS_RELEASE_2025Q3.jsonl.gz"))

rows = []
for record in records:
    ticker = record["focal_assets"][0]["identifier_value"]
    for i, fact in enumerate(facts_from_disclosure(record), start=1):
        rows.append({"ticker": ticker, "n": i, "fact": fact})

pd.set_option("display.max_colwidth", 100)
display(pd.DataFrame(rows).set_index(["ticker", "n"]))

## 2. The artifact behind the pointer

Internally the pipeline publishes a small JSON artifact per event and the competition
dereferences it. Its shape is worth seeing, because it carries one field the disclosure
does not: `parse_note`, which records how cleanly the model's output parsed.

`facts` can legitimately be `null` — see section 6.

In [ ]:
artifact = json.loads((SAMPLE_DIR / "summary_sample.json").read_text())

print("event_id  :", artifact["event_id"])
print("parse_note:", artifact["response"]["parse_note"])
print("ticker    :", artifact["metadata"]["ticker"])
print("model meta:", artifact["metadata"]["type"])
print()
for i, fact in enumerate(artifact["response"]["facts"], start=1):
    print(f"{i:2d}. {fact}")

## 3. What the model actually saw

This is the part that most shapes what the facts can possibly contain.

`format_transcript` renders the raw transcript to markdown, and the **only** metadata it
prepends is the transcript's own header title. No ticker, no event date, no fiscal
period, no consensus estimate, and no market data are injected separately. There is also
**no truncation, chunking, or token budgeting** — the entire call goes into a single user
message.

So the summarizer knows what was said on the call, plus whatever the title reveals, and
nothing else.

In [ ]:
transcript = json.loads((SAMPLE_DIR / "transcript_sample.json").read_text())
transcript_md = format_transcript(transcript)

print(f"{len(transcript['components'])} components -> {len(transcript_md):,} characters\n")
print(transcript_md[:900] + "\n...")

## 4. The exact prompt

Printed in full — this is the literal text the model receives.

Note what the instructions ask for: single-sentence, **quantified**, investor-relevant,
drawn only from the transcript, and explicitly *not* verbatim quotation. The framing
target is anticipating the stock's reaction, which is why the facts skew toward numbers,
guidance, and surprises rather than narrative summary.

In [ ]:
params = build_request(transcript_md)

print("=" * 78)
print("SYSTEM")
print("=" * 78)
print(params["system"])
print()
print("=" * 78)
print("USER  (transcript body elided)")
print("=" * 78)
user = params["messages"][0]["content"]
head, _, tail = user.partition("<transcript>")
print(head + "<transcript>")
print(f"    ... {len(transcript_md):,} characters of rendered transcript ...")
print("</transcript>")

## 5. The request surface

Two things stand out.

**Extended thinking is on.** The model reasons before answering, and on current models
that is *adaptive* thinking at `effort="high"` — no fixed token budget. Thinking is a
large part of why the extraction is nondeterministic.

**There is no structured-output machinery.** No `tools`, no JSON schema, no
`response_format`, no `temperature`, no stop sequences. The ten-fact JSON shape is
requested in prose and then *validated after the fact*. That is a deliberate design
choice with a visible consequence — section 6.

In [ ]:
display(pd.DataFrame(MODEL_CONFIGS).T.rename_axis("model"))

surface = {k: v for k, v in params.items() if k != "messages"}
surface["messages"] = f"[1 user message, {len(params['messages'][0]['content']):,} chars]"
print(json.dumps(surface, indent=2))

## 6. How extraction failures surface

Because structure is enforced by prompt text rather than by the API, the response can
deviate. The pipeline applies only **safe, non-corrupting** repairs — it never fabricates
a fact, never splits one, and never pads to reach ten. Anything it cannot recover safely
becomes a failure, and the published artifact carries `"facts": null` rather than a
partial list.

The `parse_note` vocabulary below is exactly what you may observe.

In [ ]:
ten = [f"Fact {i} with revenue of ${i}00 million." for i in range(1, 11)]
payload = json.dumps({"facts": ten})

scenarios = {
    "clean output": payload,
    "wrapped in fences": "```json\n" + payload + "\n```",
    "trailing commentary": payload + "\n\nHope that helps!",
    "missing closing ]": '{"facts": [' + ", ".join(json.dumps(f) for f in ten) + "}",
    "returned 12 facts": json.dumps({"facts": [*ten, "Extra A.", "Extra B."]}),
    "returned 8 facts": json.dumps({"facts": ten[:8]}),
    "one empty fact": json.dumps({"facts": ["", *ten[1:]]}),
    "truncated mid-sentence": payload[:-40],
    "refusal / prose only": "I'm not able to summarize that transcript.",
}

rows = []
for label, raw in scenarios.items():
    facts, note = recover_facts(raw)
    rows.append(
        {
            "scenario": label,
            "parse_note": note,
            "facts recovered": "—" if facts is None else len(facts),
            "published": "facts" if facts is not None else "null",
        }
    )

display(pd.DataFrame(rows).set_index("scenario"))

## 7. Run it yourself (optional, needs an Anthropic key)

This section calls the real API on the synthetic transcript above. It is **not** a
competition credential and nothing else in this repo needs it — set `ANTHROPIC_API_KEY`
in `.env` only if you want to watch the extraction happen.

Skipped automatically when the key is absent, so the notebook still runs top-to-bottom.

In [ ]:
if not config.has_anthropic:
    print("Sample mode: no ANTHROPIC_API_KEY set. Skipping the live summarization call.")
else:
    result = summarize_transcript(transcript_md, config.anthropic_client())

    print(f"parse_note : {result.parse_note}")
    print(f"stop_reason: {result.stop_reason}")
    print(f"tokens     : {result.input_tokens:,} in / {result.output_tokens:,} out")
    print()
    if result.ok:
        for i, fact in enumerate(result.facts, start=1):
            print(f"{i:2d}. {fact}")
    else:
        print("Extraction failed; production would publish facts: null here.")
        print(result.raw_response[:500])

    if result.thinking_summary:
        print("\n--- summarized thinking ---")
        print(result.thinking_summary[:800])

## What to take away

- The facts are a **quantified, investor-framed distillation of the call and nothing
  else** — no market data, no estimates, no cross-company context. Anything beyond the
  call is yours to add.
- They are ordered by the model's judgment of relevance, which is why over-length
  responses are trimmed from the tail.
- Extraction can fail, and a failure is visible rather than silently patched. Build for
  the case where an event's facts are absent.
- Running this yourself will **not** reproduce a competition disclosure: different
  transcript, and a nondeterministic call.

Next: [`01_historical_archive.ipynb`](01_historical_archive.ipynb) turns these events into
a scored analysis frame with the competition's exact scoring transform, and
[`03_earnings_previews.ipynb`](03_earnings_previews.ipynb) covers the *other* disclosure
item — the pre-release earnings preview.